# Run Script Generator V2 Experiments 1-18

This notebook is a clean copy inspired by `run_script_generator_v2_addendum.ipynb`; the original file is not modified.

Step-by-step shape of the system:

1. Keep the addendum's helper functions for TOML/job creation and downstream infer/UDB/3D UMAP work.
2. Select one model per run by setting `MODEL_NAME` to `HyraxAutoencoderV2`, `ImageDCAE`, or `SimCLR`.
3. Start each run with `EXPERIMENT_STAGE = "optimizer_grid"`; this activates only experiments `_1`-`_8` and does not require a baseline.
4. After reviewing `_1`-`_8`, keep the same `RUN_NUMBER`, set `EXPERIMENT_STAGE = "variants"`, and set `BASELINE_EXPERIMENT_NUMBER` to the selected experiment. This activates only `_9`-`_18`.
5. The `_9`-`_18` variants inherit their optimizer and learning rate from the selected `_1`-`_8` baseline.
6. `SimCLR` supports only the `optimizer_grid` stage (`_1`-`_8`).
7. Change `RUN_NUMBER` for each new model run; experiment suffixes intentionally restart within each run.
8. Allow more experiments by adding entries to `EXTRA_EXPERIMENTS` instead of rewriting generator code.

`ImageDCAE` is Hyrax's dynamically sized DCAE, not the fixed three-channel `HSCDCAE`. Its skip-connected decoder makes it useful for comparison, but reconstruction quality alone may not imply an equally informative latent embedding.


In [1]:

import os
import re
import time
from importlib import import_module
from pathlib import Path

import tomlkit

try:
    save_umap_json = import_module("hyrax.3d_viz.save_umap_to_json").save_umap_json
except Exception:
    save_umap_json = None


DEFAULT_BASE_DIRECTORY = Path("/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs")
DEFAULT_DATA_DIR = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images_120/"
DEFAULT_RESULTS_DIR = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/results/"
DEFAULT_FILTERS = ["g", "r", "i", "z", "y"]


def _default_output_dir(run_number, base_directory=DEFAULT_BASE_DIRECTORY):
    return Path(base_directory) / f"run{run_number}"


def _load_toml(path):
    with open(path, "r", encoding="utf-8") as handle:
        return tomlkit.load(handle)


def _write_toml(path, document):
    with open(path, "w", encoding="utf-8") as handle:
        handle.write(tomlkit.dumps(document))


def _iter_dataset_wrappers(data_request_section):
    if not isinstance(data_request_section, dict):
        return
    for dataset_wrapper in data_request_section.values():
        if isinstance(dataset_wrapper, dict) and "data" in dataset_wrapper:
            yield dataset_wrapper


def _find_results_dir_by_run_name(results_root, run_name, expected_suffix=None):
    results_root = Path(results_root)
    if not results_root.exists():
        return None

    for candidate in sorted(results_root.iterdir(), reverse=True):
        if not candidate.is_dir():
            continue
        if expected_suffix and expected_suffix not in candidate.name:
            continue
        runtime_config = candidate / "runtime_config.toml"
        if not runtime_config.exists():
            continue
        try:
            config = _load_toml(runtime_config)
        except Exception:
            continue
        if config.get("train", {}).get("run_name") == run_name:
            return candidate

    return None


In [2]:
from copy import deepcopy


def _default_model_config_v2(model_name):
    """Return model-specific config blocks for generated runtime TOMLs."""
    if model_name == "SimCLR":
        return {
            "SimCLR": {
                "projection_dimension": 128,
                "temperature": 0.5,
                "horizontal_flip_probability": 0.5,
                "color_jitter_params": [0.8, 0.8, 0.8, 0.2],
                "color_jitter_probability": 0.8,
                "grayscale_probability": 0.2,
                "gaussian_blur_kernel_size": 9,
                "gaussian_blur_sigma_range": [0.1, 2.0],
            }
        }

    if model_name == "HyraxAutoencoderV2":
        return {
            "HyraxAutoencoderV2": {
                "base_channel_size": 32,
                "latent_dim": 64,
                "final_layer": "tanh",
            }
        }

    if model_name == "ImageDCAE":
        return {
            "ImageDCAE": {
                "base_channel_size": 32,
                "latent_dim": 512,
                "final_layer": "identity",
            }
        }

    return {}


def _default_criterion_config_v2(model_name):
    """Return criterion config for generated runtime TOMLs."""
    if model_name == "SimCLR":
        return {
            # SimCLR defines NTXentLoss internally from model.SimCLR.temperature.
            "name": "",
            "band_loss_reduction": "mean",
        }

    return {
        "name": "torch.nn.MSELoss",
        "band_loss_reduction": "mean",
    }


def _deep_update(base, updates):
    """Recursively merge updates into base and return base."""
    for key, value in (updates or {}).items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            _deep_update(base[key], value)
        else:
            base[key] = deepcopy(value)
    return base


def optimizer_config_v2(optimizer, lr, momentum=False):
    """Create one optimizer spec in the format expected by the TOML writer."""
    return {
        "optimizer": optimizer,
        "lr": lr,
        "momentum": bool(momentum),
    }


OPTIMIZER_GRID_V2 = {
    1: optimizer_config_v2("torch.optim.SGD", 0.01, momentum=True),
    2: optimizer_config_v2("torch.optim.SGD", 0.1, momentum=True),
    3: optimizer_config_v2("torch.optim.SGD", 0.001, momentum=True),
    4: optimizer_config_v2("torch.optim.SGD", 0.0001, momentum=True),
    5: optimizer_config_v2("torch.optim.Adam", 0.01),
    6: optimizer_config_v2("torch.optim.Adam", 0.1),
    7: optimizer_config_v2("torch.optim.Adam", 0.001),
    8: optimizer_config_v2("torch.optim.Adam", 0.0001),
}


DEFAULT_BASELINE_EXPERIMENT_NUMBER = 7
DEFAULT_EXPERIMENT_STAGE = "optimizer_grid"
EXPERIMENT_STAGES_V2 = ("optimizer_grid", "variants")
SUPPORTED_MODEL_NAMES_V2 = ("HyraxAutoencoderV2", "ImageDCAE", "SimCLR")
AUTOENCODER_MODEL_NAMES_V2 = ("HyraxAutoencoderV2", "ImageDCAE")


def _spec(description, optimizer_config=None, overrides=None, baseline_experiment=None):
    return {
        "description": description,
        "optimizer_config": deepcopy(optimizer_config),
        "overrides": deepcopy(overrides or {}),
        "baseline_experiment": baseline_experiment,
    }


def build_optimizer_experiment_specs_v2(model_name="HyraxAutoencoderV2"):
    """Build experiments 1-8 for an autoencoder optimizer/lr sweep."""
    if model_name not in AUTOENCODER_MODEL_NAMES_V2:
        raise ValueError(
            f"build_optimizer_experiment_specs_v2 requires one of {AUTOENCODER_MODEL_NAMES_V2}; "
            f"received {model_name!r}"
        )

    return {
        idx: _spec(
            description=f"{cfg['optimizer'].split('.')[-1]} lr={cfg['lr']}",
            optimizer_config=cfg,
            overrides={"model_name": model_name},
        )
        for idx, cfg in OPTIMIZER_GRID_V2.items()
    }


def build_variant_experiment_specs_v2(
    model_name,
    baseline_experiment_number,
    optimizer_specs=None,
):
    """Build experiments 9-18 from one selected optimizer experiment."""
    if model_name not in AUTOENCODER_MODEL_NAMES_V2:
        raise ValueError(
            f"The variants stage requires one of {AUTOENCODER_MODEL_NAMES_V2}; "
            f"received {model_name!r}"
        )
    if baseline_experiment_number is None:
        raise ValueError(
            "Set BASELINE_EXPERIMENT_NUMBER to an experiment from 1 through 8 "
            "before using EXPERIMENT_STAGE='variants'."
        )

    baseline_experiment_number = int(baseline_experiment_number)
    optimizer_specs = optimizer_specs or build_optimizer_experiment_specs_v2(model_name)
    if baseline_experiment_number not in range(1, 9) or baseline_experiment_number not in optimizer_specs:
        raise ValueError(
            f"BASELINE_EXPERIMENT_NUMBER must be from 1 through 8; "
            f"received {baseline_experiment_number}."
        )
    baseline_label = f"baseline _{baseline_experiment_number}"
    baseline = deepcopy(optimizer_specs[baseline_experiment_number]["optimizer_config"])
    return {
        9: _spec(f"{baseline_label} with batch_size=512", baseline, {"model_name": model_name, "batch_size": 512}, baseline_experiment_number),
        10: _spec(f"{baseline_label} with batch_size=128", baseline, {"model_name": model_name, "batch_size": 128}, baseline_experiment_number),
        11: _spec(
            f"{baseline_label} with final_layer=arcsinh and transform=arcsinh",
            baseline,
            {
                "model_name": model_name,
                "model_config_overrides": {model_name: {"final_layer": "arcsinh"}},
                "transform": "arcsinh",
            },
            baseline_experiment_number,
        ),
        12: _spec(f"{baseline_label} with filters=[g,r,i,z]", baseline, {"model_name": model_name, "filters": ["g", "r", "i", "z"]}, baseline_experiment_number),
        13: _spec(f"{baseline_label} with latent_dim=128", baseline, {"model_name": model_name, "model_config_overrides": {model_name: {"latent_dim": 128}}}, baseline_experiment_number),
        14: _spec(f"{baseline_label} with latent_dim=256", baseline, {"model_name": model_name, "model_config_overrides": {model_name: {"latent_dim": 256}}}, baseline_experiment_number),
        15: _spec(f"{baseline_label} with latent_dim=1024", baseline, {"model_name": model_name, "model_config_overrides": {model_name: {"latent_dim": 1024}}}, baseline_experiment_number),
        16: _spec(f"{baseline_label} with base_channel_size=16", baseline, {"model_name": model_name, "model_config_overrides": {model_name: {"base_channel_size": 16}}}, baseline_experiment_number),
        17: _spec(f"{baseline_label} with base_channel_size=64", baseline, {"model_name": model_name, "model_config_overrides": {model_name: {"base_channel_size": 64}}}, baseline_experiment_number),
        18: _spec(f"{baseline_label} with crop_to=[100,100]", baseline, {"model_name": model_name, "crop_to": [100, 100]}, baseline_experiment_number),
    }


def build_experiment_specs_v2(
    baseline_experiment_number=DEFAULT_BASELINE_EXPERIMENT_NUMBER,
    extra_experiments=None,
    model_name="HyraxAutoencoderV2",
):
    """Build the complete 1-18 autoencoder spec map for compatibility."""
    optimizer_specs = build_optimizer_experiment_specs_v2(model_name)
    variant_specs = build_variant_experiment_specs_v2(
        model_name=model_name,
        baseline_experiment_number=baseline_experiment_number,
        optimizer_specs=optimizer_specs,
    )
    specs = {**optimizer_specs, **variant_specs}
    baseline = deepcopy(optimizer_specs[int(baseline_experiment_number)]["optimizer_config"])

    for idx, spec in (extra_experiments or {}).items():
        spec = deepcopy(spec)
        baseline_idx = spec.get("baseline_experiment")
        optimizer_config = spec.get("optimizer_config")
        if optimizer_config is None and baseline_idx is not None:
            optimizer_config = specs[baseline_idx]["optimizer_config"]
        if optimizer_config is None:
            optimizer_config = baseline

        specs[int(idx)] = _spec(
            description=spec.get("description", f"custom experiment {idx}"),
            optimizer_config=optimizer_config,
            overrides=_deep_update({"model_name": model_name}, spec.get("overrides", {})),
            baseline_experiment=baseline_idx,
        )

    return dict(sorted(specs.items()))


def build_simclr_experiment_specs_v2(
    start_number=1,
    optimizer_grid=None,
    filters=None,
    model_config_overrides=None,
):
    """Build the SimCLR optimizer sweep for a SimCLR-only run."""
    optimizer_grid = optimizer_grid or OPTIMIZER_GRID_V2
    filters = list(filters) if filters is not None else ["g", "r", "i"]
    model_config_overrides = deepcopy(model_config_overrides or {})
    specs = {}

    for offset, (optimizer_idx, cfg) in enumerate(sorted(optimizer_grid.items())):
        experiment_number = int(start_number) + offset
        optimizer_label = cfg["optimizer"].split(".")[-1]
        specs[experiment_number] = _spec(
            description=f"SimCLR {optimizer_label} lr={cfg['lr']} (optimizer grid _{optimizer_idx})",
            optimizer_config=cfg,
            overrides={
                "model_name": "SimCLR",
                "filters": filters,
                "model_config_overrides": {"SimCLR": model_config_overrides},
            },
        )

    return specs


def build_model_experiment_plan_v2(
    model_name,
    experiment_stage=DEFAULT_EXPERIMENT_STAGE,
    baseline_experiment_number=None,
    simclr_filters=None,
):
    """Return all reference specs and the experiment numbers active in this stage."""
    if model_name not in SUPPORTED_MODEL_NAMES_V2:
        raise ValueError(f"Unsupported model {model_name!r}; choose one of {SUPPORTED_MODEL_NAMES_V2}")
    if experiment_stage not in EXPERIMENT_STAGES_V2:
        raise ValueError(
            f"Unsupported EXPERIMENT_STAGE {experiment_stage!r}; "
            f"choose one of {EXPERIMENT_STAGES_V2}."
        )
    if model_name == "SimCLR":
        if experiment_stage == "variants":
            raise ValueError("SimCLR supports only EXPERIMENT_STAGE='optimizer_grid'.")
        specs = build_simclr_experiment_specs_v2(start_number=1, filters=simclr_filters)
        return specs, sorted(specs)

    optimizer_specs = build_optimizer_experiment_specs_v2(model_name)
    if experiment_stage == "optimizer_grid":
        return optimizer_specs, sorted(optimizer_specs)

    variant_specs = build_variant_experiment_specs_v2(
        model_name=model_name,
        baseline_experiment_number=baseline_experiment_number,
        optimizer_specs=optimizer_specs,
    )
    return {**optimizer_specs, **variant_specs}, sorted(variant_specs)


def build_model_experiment_specs_v2(
    model_name,
    baseline_experiment_number=DEFAULT_BASELINE_EXPERIMENT_NUMBER,
    simclr_filters=None,
):
    """Build the complete model spec map for backward compatibility."""
    if model_name == "SimCLR":
        return build_simclr_experiment_specs_v2(start_number=1, filters=simclr_filters)
    return build_experiment_specs_v2(
        model_name=model_name,
        baseline_experiment_number=baseline_experiment_number,
    )


def build_extra_experiment_specs_v2(
    extra_experiments,
    existing_specs,
    default_baseline_experiment=DEFAULT_BASELINE_EXPERIMENT_NUMBER,
):
    """Build custom experiments, inheriting model overrides from their baseline when requested."""
    if not extra_experiments:
        return {}

    existing_specs = existing_specs or build_optimizer_experiment_specs_v2()
    default_optimizer_config = None
    if default_baseline_experiment is not None:
        default_baseline_experiment = int(default_baseline_experiment)
        if default_baseline_experiment not in existing_specs:
            raise KeyError(f"Default baseline experiment _{default_baseline_experiment} is not defined")
        default_optimizer_config = existing_specs[default_baseline_experiment]["optimizer_config"]
    extra_specs = {}

    for idx, spec in extra_experiments.items():
        spec = deepcopy(spec)
        baseline_idx = spec.get("baseline_experiment")
        baseline_spec = None
        if baseline_idx is not None:
            baseline_idx = int(baseline_idx)
            if baseline_idx not in existing_specs:
                raise KeyError(f"Baseline experiment _{baseline_idx} is not defined")
            baseline_spec = existing_specs[baseline_idx]

        optimizer_config = spec.get("optimizer_config")
        if optimizer_config is None and baseline_spec is not None:
            optimizer_config = baseline_spec["optimizer_config"]
        if optimizer_config is None:
            if default_optimizer_config is None:
                raise ValueError(
                    f"Custom experiment _{idx} must provide optimizer_config or baseline_experiment "
                    "when BASELINE_EXPERIMENT_NUMBER is None."
                )
            optimizer_config = default_optimizer_config

        overrides = deepcopy(baseline_spec.get("overrides", {})) if baseline_spec is not None else {}
        _deep_update(overrides, spec.get("overrides", {}))

        extra_specs[int(idx)] = _spec(
            description=spec.get("description", f"custom experiment {idx}"),
            optimizer_config=optimizer_config,
            overrides=overrides,
            baseline_experiment=baseline_idx,
        )

    return extra_specs


def print_experiment_specs_v2(experiment_specs=None):
    """Print a compact summary of the experiment plan before writing files."""
    experiment_specs = experiment_specs or build_experiment_specs_v2()
    for idx, spec in experiment_specs.items():
        baseline = spec.get("baseline_experiment")
        baseline_text = f" baseline=_{baseline}" if baseline else ""
        print(f"_{idx}: {spec['description']}{baseline_text}")


def _resolve_model_config(model_name, base_model_config, overrides):
    model_config = deepcopy(base_model_config if base_model_config is not None else _default_model_config_v2(model_name))
    return _deep_update(model_config, overrides or {})


def create_training_files_v2(
    run_number,
    filter_catalog,
    crop_to=None,
    output_dir=None,
    data_dir=DEFAULT_DATA_DIR,
    results_dir=DEFAULT_RESULTS_DIR,
    model_name="HyraxAutoencoderV2",
    model_config=None,
    batch_size=256,
    epochs=20,
    dataset_class="FitsImageDataSet",
    object_id_column_name="object_id",
    filters=None,
    transform="tanh",
    baseline_experiment_number=None,
    experiment_stage=DEFAULT_EXPERIMENT_STAGE,
    experiment_specs=None,
    experiment_numbers=None,
):
    """Create training TOMLs and SLURM job files for selected experiments.

    By default, this writes only the 1-8 optimizer grid. Set experiment_stage
    to 'variants' and provide a 1-8 baseline to write experiments 9-18.
    """
    output_dir = Path(output_dir) if output_dir else _default_output_dir(run_number)
    crop_to = list(crop_to) if crop_to is not None else [150, 150]
    filters = (
        list(filters)
        if filters is not None
        else (["g", "r", "i"] if model_name == "SimCLR" else list(DEFAULT_FILTERS))
    )
    model_config = deepcopy(model_config if model_config is not None else _default_model_config_v2(model_name))
    if experiment_specs is None:
        experiment_specs, default_experiment_numbers = build_model_experiment_plan_v2(
            model_name=model_name,
            experiment_stage=experiment_stage,
            baseline_experiment_number=baseline_experiment_number,
            simclr_filters=filters if model_name == "SimCLR" else None,
        )
    else:
        default_experiment_numbers = sorted(experiment_specs)
    experiment_numbers = (
        list(experiment_numbers) if experiment_numbers is not None else default_experiment_numbers
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    for file_number in experiment_numbers:
        if file_number not in experiment_specs:
            raise KeyError(f"Experiment _{file_number} is not defined in experiment_specs")

        spec = experiment_specs[file_number]
        overrides = spec.get("overrides", {})
        current_model_name = overrides.get("model_name", model_name)
        current_model_config = _resolve_model_config(
            current_model_name,
            model_config if current_model_name == model_name else None,
            overrides.get("model_config_overrides"),
        )

        create_toml_file_v2(
            run_number=run_number,
            file_number=file_number,
            optimizer_config=spec["optimizer_config"],
            filter_catalog=filter_catalog,
            crop_to=overrides.get("crop_to", crop_to),
            output_dir=output_dir,
            data_dir=overrides.get("data_dir", data_dir),
            results_dir=overrides.get("results_dir", results_dir),
            model_name=current_model_name,
            model_config=current_model_config,
            batch_size=overrides.get("batch_size", batch_size),
            epochs=overrides.get("epochs", epochs),
            dataset_class=overrides.get("dataset_class", dataset_class),
            object_id_column_name=overrides.get("object_id_column_name", object_id_column_name),
            filters=overrides.get("filters", filters),
            transform=overrides.get("transform", transform),
        )
        create_job_file_v2(run_number, file_number, output_dir)


def create_toml_file_v2(
    run_number,
    file_number,
    optimizer_config,
    filter_catalog,
    crop_to,
    output_dir,
    data_dir,
    results_dir,
    model_name,
    model_config,
    batch_size,
    epochs,
    dataset_class,
    object_id_column_name,
    filters,
    transform,
):
    """Create a Hyrax runtime config that matches the current codebase."""
    optimizer_name = optimizer_config["optimizer"]
    run_name = f"run{run_number}_{file_number}"

    document = tomlkit.document()
    document["general"] = {
        "dev_mode": False,
        "log_level": "debug",
        "data_dir": str(data_dir),
        "results_dir": str(results_dir),
    }

    document["model"] = {"name": model_name}
    document["model"].update(deepcopy(model_config))

    document["criterion"] = _default_criterion_config_v2(model_name)

    document["optimizer"] = {"name": optimizer_name}
    document[optimizer_name] = {"lr": optimizer_config["lr"]}
    if optimizer_config["momentum"]:
        document[optimizer_name]["momentum"] = 0.9

    document["train"] = {
        "weights_filename": "example_model.pth",
        "epochs": epochs,
        "resume": False,
        "split": "train",
        "experiment_name": f"run{run_number}",
        "run_name": run_name,
    }

    document["data_request"] = {
        "train": {
            "data": {
                "dataset_class": dataset_class,
                "data_location": str(data_dir),
                "fields": ["image"],
                "primary_id_field": "object_id",
            }
        },
        "infer": {
            "data": {
                "dataset_class": dataset_class,
                "data_location": str(data_dir),
                "fields": ["image"],
                "primary_id_field": "object_id",
            }
        },
    }

    document["data_set"] = {
        "name": dataset_class,
        "object_id_column_name": object_id_column_name,
        "use_cache": True,
        "preload_cache": True,
        "seed": 1,
        "train_size": 0.8,
        "validate_size": 0.1,
        "test_size": 0.1,
        "filter_catalog": str(filter_catalog),
        "filters": list(filters),
        "transform": transform,
        "crop_to": list(crop_to),
    }

    document["data_loader"] = {"batch_size": batch_size}

    output_path = Path(output_dir) / f"train{run_number}_{file_number}.toml"
    _write_toml(output_path, document)
    print(f"Created {output_path}")


def create_job_file_v2(run_number, file_number, output_dir):
    """Create a SLURM job file for training."""
    output_dir = Path(output_dir)
    toml_path = output_dir / f"train{run_number}_{file_number}.toml"
    job_content = f"""#!/bin/bash
#
#SBATCH --job-name=t{run_number}_{file_number}
#SBATCH --output=train{run_number}_{file_number}.txt
#SBATCH --account=bemi-delta-gpu
#SBATCH --partition=gpuA40x4
#SBATCH --nodes=1
#SBATCH --cpus-per-gpu=5
#SBATCH --mem=50G
#SBATCH --gpus=1
#SBATCH --time=1:00:00

# source /mmfs1/home/aritrag/.bashrc
# conda activate hyrax
hyrax train --runtime-config={toml_path}
"""
    output_path = output_dir / f"train{run_number}_{file_number}.sh"
    output_path.write_text(job_content, encoding="utf-8")
    print(f"Created {output_path}")


def submit_training_jobs_v2(run_number, job_numbers, end_num=None, base_directory=DEFAULT_BASE_DIRECTORY):
    """Submit selected training jobs.

    Backward-compatible forms:
      submit_training_jobs_v2(run_number, 1, 8)
      submit_training_jobs_v2(run_number, [1, 2, 3])
      submit_training_jobs_v2(run_number, range(9, 19))
    """
    run_dir = Path(base_directory) / f"run{run_number}"
    os.chdir(run_dir)

    if end_num is not None:
        job_numbers = range(int(job_numbers), int(end_num) + 1)

    for offset, i in enumerate(job_numbers):
        job_file = f"train{run_number}_{i}.sh"
        cmd = f"sbatch {job_file}"
        print(f"Submitting: {cmd}")
        os.system(cmd)
        if offset == 0:
            time.sleep(10)


def copy_training_configs_v2(source_dir, target_dir, old_run_num, new_run_num, new_data_dir=None, new_filter_catalog_root=None):
    """Copy generated training configs/jobs and update run identifiers and key paths."""
    source_path = Path(source_dir)
    target_path = Path(target_dir)
    target_path.mkdir(parents=True, exist_ok=True)

    copied = 0
    for toml_file in sorted(source_path.glob(f"train{old_run_num}_*.toml")):
        suffix = toml_file.stem.split("_")[-1]
        new_toml_name = f"train{new_run_num}_{suffix}.toml"
        new_job_name = f"train{new_run_num}_{suffix}.sh"

        config = _load_toml(toml_file)
        config.setdefault("train", {})
        config["train"]["experiment_name"] = f"run{new_run_num}"
        config["train"]["run_name"] = f"run{new_run_num}_{suffix}"
        config["train"]["weights_filename"] = config["train"].get("weights_filename", "example_model.pth")
        config["general"]["log_level"] = "debug"

        if new_data_dir is not None:
            config["general"]["data_dir"] = str(new_data_dir)
            for split_name in ("train", "infer", "validate"):
                split_config = config.get("data_request", {}).get(split_name)
                for dataset_wrapper in _iter_dataset_wrappers(split_config):
                    dataset_wrapper["data"]["data_location"] = str(new_data_dir)

        if new_filter_catalog_root is not None and config.get("data_set", {}).get("filter_catalog"):
            current_name = Path(str(config["data_set"]["filter_catalog"])).name
            config["data_set"]["filter_catalog"] = str(Path(new_filter_catalog_root) / current_name)

        _write_toml(target_path / new_toml_name, config)

        job_file = source_path / f"train{old_run_num}_{suffix}.sh"
        if job_file.exists():
            job_content = job_file.read_text(encoding="utf-8")
            job_content = job_content.replace(f"--job-name=t{old_run_num}_{suffix}", f"--job-name=t{new_run_num}_{suffix}")
            job_content = job_content.replace(f"--output=train{old_run_num}_{suffix}.txt", f"--output=train{new_run_num}_{suffix}.txt")
            job_content = job_content.replace(f"run{old_run_num}/train{old_run_num}_{suffix}.toml", f"run{new_run_num}/train{new_run_num}_{suffix}.toml")
            (target_path / new_job_name).write_text(job_content, encoding="utf-8")

        copied += 1

    print(f"Copied {copied} training configurations from run{old_run_num} to run{new_run_num}")


def extract_model_directory_v2(train_output_file):
    """Return the trained weights path, using log parsing first and run-name matching as fallback."""
    train_output_file = Path(train_output_file)
    if not train_output_file.exists():
        raise FileNotFoundError(f"Training output file not found: {train_output_file}")

    content = train_output_file.read_text(encoding="utf-8", errors="replace")
    patterns = [
        r"Latest checkpoint saved as: (.+)/checkpoint_epoch_\d+\.pt",
        r"Best metric checkpoint saved as: (.+)/checkpoint_[^/\s]+\.pt",
        r"Exported model to ONNX format: (.+)/[^/\s]+\.onnx",
    ]
    for pattern in patterns:
        match = re.search(pattern, content)
        if match:
            return str(Path(match.group(1)) / "example_model.pth")

    sibling_toml = train_output_file.with_suffix(".toml")
    if sibling_toml.exists():
        config = _load_toml(sibling_toml)
        results_root = config.get("general", {}).get("results_dir")
        run_name = config.get("train", {}).get("run_name")
        weights_filename = config.get("train", {}).get("weights_filename", "example_model.pth")
        if results_root and run_name:
            results_dir = _find_results_dir_by_run_name(results_root, run_name, expected_suffix="-train-")
            if results_dir is not None:
                candidate = results_dir / weights_filename
                if candidate.exists():
                    return str(candidate)

    raise ValueError(f"Could not determine model weights path from {train_output_file}")


def create_infer_scripts_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY):
    """Generate infer TOMLs/jobs from sibling train files."""
    run_dir = Path(base_directory) / f"run{x}"
    results = []

    for y in y_values:
        train_name = f"train{x}_{y}"
        infer_name = f"infer{x}_{y}"
        train_toml = run_dir / f"{train_name}.toml"
        train_job = run_dir / f"{train_name}.sh"
        train_output = run_dir / f"{train_name}.txt"

        if not train_toml.exists():
            print(f"Warning: {train_toml} not found, skipping...")
            continue

        try:
            model_weights_path = extract_model_directory_v2(train_output)
        except (FileNotFoundError, ValueError) as exc:
            print(f"Error processing {train_name}: {exc}")
            continue

        infer_toml = run_dir / f"{infer_name}.toml"
        config = _load_toml(train_toml)
        config["infer"] = {"model_weights_file": str(model_weights_path), "split": False}
        _write_toml(infer_toml, config)

        if train_job.exists():
            infer_job = run_dir / f"{infer_name}.sh"
            job_content = train_job.read_text(encoding="utf-8")
            job_content = job_content.replace(f"--job-name=t{x}_{y}", f"--job-name=i{x}_{y}")
            job_content = job_content.replace(f"--output=train{x}_{y}.txt", f"--output=infer{x}_{y}.txt")
            job_content = job_content.replace(f"train{x}_{y}.toml", f"infer{x}_{y}.toml")
            job_content = job_content.replace("hyrax train", "hyrax infer")
            infer_job.write_text(job_content, encoding="utf-8")
            results.append((infer_toml, infer_job))
            print(f"Created infer scripts for {train_name}")
        else:
            print(f"Warning: {train_job} not found for {train_name}")

    return results


def submit_infer_jobs_v2(run_number, job_numbers, base_directory=DEFAULT_BASE_DIRECTORY):
    """Submit selected infer jobs."""
    run_dir = Path(base_directory) / f"run{run_number}"
    os.chdir(run_dir)
    for i in job_numbers:
        job_file = f"infer{run_number}_{i}.sh"
        cmd = f"sbatch {job_file}"
        print(f"Submitting: {cmd}")
        os.system(cmd)


def extract_inference_directory_v2(infer_output_file):
    """Extract the inference results directory from infer output logs."""
    infer_output_file = Path(infer_output_file)
    if not infer_output_file.exists():
        raise FileNotFoundError(f"Inference output file not found: {infer_output_file}")

    content = infer_output_file.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"Saving inference results at: (.+)", content)
    if match:
        return match.group(1).strip().rstrip("/") + "/"

    raise ValueError(f"Could not find inference directory in {infer_output_file}")


In [3]:

def create_udb_scripts_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY):
    """Generate UDB TOMLs/jobs from infer artifacts."""
    run_dir = Path(base_directory) / f"run{x}"
    results = []

    for y in y_values:
        infer_name = f"infer{x}_{y}"
        udb_name = f"udb{x}_{y}"
        infer_toml = run_dir / f"{infer_name}.toml"
        infer_job = run_dir / f"{infer_name}.sh"
        infer_output = run_dir / f"{infer_name}.txt"

        if not infer_toml.exists():
            print(f"Warning: {infer_toml} not found, skipping...")
            continue

        try:
            inference_dir = extract_inference_directory_v2(infer_output)
        except (FileNotFoundError, ValueError) as exc:
            print(f"Error processing {infer_name}: {exc}")
            continue

        udb_toml = run_dir / f"{udb_name}.toml"
        config = _load_toml(infer_toml)
        config["results"] = {"inference_dir": inference_dir}
        config["vector_db"] = {"name": "chromadb", "infer_results_dir": inference_dir}
        config["umap"] = {
            "fit_sample_size": 5000,
            "save_fit_umap": False,
            "parallel": True,
            "name": "umap.UMAP",
            "UMAP": {"n_components": 2, "n_neighbors": 15},
        }
        _write_toml(udb_toml, config)

        if infer_job.exists():
            udb_job = run_dir / f"{udb_name}.sh"
            job_content = infer_job.read_text(encoding="utf-8")
            job_content = job_content.replace(f"--job-name=i{x}_{y}", f"--job-name=u{x}_{y}")
            job_content = job_content.replace(f"--output=infer{x}_{y}.txt", f"--output=udb{x}_{y}.txt")
            job_content = job_content.replace(f"infer{x}_{y}.toml", f"udb{x}_{y}.toml")
            input_dir = inference_dir.rstrip("/")
            original = "hyrax infer --runtime-config="
            replacement = (
                f"hyrax umap --runtime-config="
                f"{{runtime}} --input-dir={input_dir}/\n"
                f"hyrax save_to_database --runtime-config={{runtime}} --input-dir={input_dir}/"
            )
            match = re.search(r"hyrax infer --runtime-config=([^\s]+)", job_content)
            if match:
                job_content = job_content.replace(
                    f"hyrax infer --runtime-config={match.group(1)}",
                    replacement.format(runtime=match.group(1)),
                )
            udb_job.write_text(job_content, encoding="utf-8")
            results.append((udb_toml, udb_job))
            print(f"Created UDB scripts for {infer_name}")
        else:
            print(f"Warning: {infer_job} not found for {infer_name}")

    return results


def submit_udb_jobs_v2(run_number, job_numbers, base_directory=DEFAULT_BASE_DIRECTORY):
    """Submit selected UDB jobs."""
    run_dir = Path(base_directory) / f"run{run_number}"
    os.chdir(run_dir)
    for i in job_numbers:
        job_file = f"udb{run_number}_{i}.sh"
        cmd = f"sbatch {job_file}"
        print(f"Submitting: {cmd}")
        os.system(cmd)


def create_3dumap_scripts_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY):
    """Generate 3D UMAP TOMLs/jobs from UDB artifacts."""
    run_dir = Path(base_directory) / f"run{x}"
    results = []

    for y in y_values:
        udb_name = f"udb{x}_{y}"
        dumap_name = f"3dumap{x}_{y}"
        udb_toml = run_dir / f"{udb_name}.toml"
        udb_job = run_dir / f"{udb_name}.sh"

        if not udb_toml.exists():
            print(f"Warning: {udb_toml} not found, skipping...")
            continue

        dumap_toml = run_dir / f"{dumap_name}.toml"
        config = _load_toml(udb_toml)
        config.setdefault("umap", {})
        config["umap"].setdefault("UMAP", {})
        config["umap"]["UMAP"]["n_components"] = 3
        _write_toml(dumap_toml, config)

        if udb_job.exists():
            dumap_job = run_dir / f"{dumap_name}.sh"
            lines = udb_job.read_text(encoding="utf-8").splitlines()
            filtered_lines = [line for line in lines if "hyrax save_to_database" not in line]
            job_content = "\n".join(filtered_lines) + "\n"
            job_content = job_content.replace(f"--job-name=u{x}_{y}", f"--job-name=3u{x}_{y}")
            job_content = job_content.replace(f"--output=udb{x}_{y}.txt", f"--output=3dumap{x}_{y}.txt")
            job_content = job_content.replace(f"udb{x}_{y}.toml", f"3dumap{x}_{y}.toml")
            dumap_job.write_text(job_content, encoding="utf-8")
            results.append((dumap_toml, dumap_job))
            print(f"Created 3D UMAP scripts for {udb_name}")
        else:
            print(f"Warning: {udb_job} not found for {udb_name}")

    return results


def submit_3dumap_jobs_v2(run_number, job_numbers, base_directory=DEFAULT_BASE_DIRECTORY):
    """Submit selected 3D UMAP jobs."""
    run_dir = Path(base_directory) / f"run{run_number}"
    os.chdir(run_dir)
    for i in job_numbers:
        job_file = f"3dumap{run_number}_{i}.sh"
        cmd = f"sbatch {job_file}"
        print(f"Submitting: {cmd}")
        os.system(cmd)


def extract_umap_directory_v2(dumap_output_file):
    """Extract the UMAP results directory from 3D UMAP output logs."""
    dumap_output_file = Path(dumap_output_file)
    if not dumap_output_file.exists():
        raise FileNotFoundError(f"3D UMAP output file not found: {dumap_output_file}")

    content = dumap_output_file.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"Saving UMAP results to (.+)", content)
    if match:
        return match.group(1).strip()

    raise ValueError(f"Could not find UMAP directory in {dumap_output_file}")


def extract_catalog_settings_v2(dumap_config_file):
    """Extract the catalog path and whether duplicate object IDs should be collapsed."""
    dumap_config_file = Path(dumap_config_file)
    if not dumap_config_file.exists():
        raise FileNotFoundError(f"3D UMAP config file not found: {dumap_config_file}")

    config = _load_toml(dumap_config_file)

    filter_catalog = config.get("data_set", {}).get("filter_catalog")
    if filter_catalog:
        return str(filter_catalog), True

    legacy_path = config.get("data_set", {}).get("astropy_table")
    if legacy_path:
        return str(legacy_path), False

    for split_name in ("train", "infer", "validate"):
        split_config = config.get("data_request", {}).get(split_name)
        for dataset_wrapper in _iter_dataset_wrappers(split_config):
            dataset_config = dataset_wrapper["data"].get("dataset_config", {})
            if isinstance(dataset_config, dict):
                filter_catalog = dataset_config.get("filter_catalog")
                if filter_catalog:
                    return str(filter_catalog), True
                astropy_table = dataset_config.get("astropy_table")
                if astropy_table:
                    return str(astropy_table), False

    raise ValueError(f"Could not find filter_catalog or astropy_table in {dumap_config_file}")


def create_3d_viz_json_v2(x, y, base_directory=DEFAULT_BASE_DIRECTORY, id_column="object_id"):
    """Create a 3D visualization JSON file from 3D UMAP results."""
    if save_umap_json is None:
        raise ImportError("Could not import save_umap_json from hyrax. Activate a Hyrax environment before running.")

    run_dir = Path(base_directory) / f"run{x}"
    dumap_name = f"3dumap{x}_{y}"
    dumap_output = run_dir / f"{dumap_name}.txt"
    dumap_config = run_dir / f"{dumap_name}.toml"

    if not dumap_output.exists():
        raise FileNotFoundError(f"3D UMAP output file not found: {dumap_output}")
    if not dumap_config.exists():
        raise FileNotFoundError(f"3D UMAP config file not found: {dumap_config}")

    umap_results_dir = extract_umap_directory_v2(dumap_output)
    fits_table_path, keep_first_match_only = extract_catalog_settings_v2(dumap_config)

    viz_dir = Path(base_directory) / "3d_viz_files"
    viz_dir.mkdir(exist_ok=True)
    output_json = viz_dir / f"umap{x}_{y}.json"

    save_umap_json(
        results_dir=umap_results_dir,
        output_json=str(output_json),
        fits_table_path=fits_table_path,
        id_column=id_column,
        keep_first_match_only=keep_first_match_only,
    )
    print(f"Successfully created {output_json}")
    return str(output_json)


def create_3d_viz_json_batch_v2(x, y_values, base_directory=DEFAULT_BASE_DIRECTORY, id_column="object_id"):
    """Create 3D visualization JSON files for multiple jobs."""
    results = []
    failed = []

    for y in y_values:
        try:
            output_json = create_3d_viz_json_v2(x, y, base_directory=base_directory, id_column=id_column)
            results.append(output_json)
        except Exception as exc:
            print(f"Failed to process 3dumap{x}_{y}: {exc}")
            failed.append(y)

    print("\n3D Visualization JSON Generation Summary:")
    print(f"  Successfully created: {len(results)} JSON files")
    print(f"  Failed: {len(failed)} files")
    if failed:
        print(f"  Failed job numbers: {failed}")

    return results


In [4]:
from pathlib import Path

BASE_DIR = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs"
RUN_NUMBER = 11
# Choose one model per run: "HyraxAutoencoderV2", "ImageDCAE", or "SimCLR".
MODEL_NAME = "ImageDCAE"
# Start with "optimizer_grid". After reviewing _1-_8, change this to "variants".
EXPERIMENT_STAGE = "variants"
# Leave this as None for optimizer_grid. For variants, set it to the selected _1-_8 experiment.
BASELINE_EXPERIMENT_NUMBER = 3

SIMCLR_FILTERS = ["g", "r", "i"]

DATA_DIR = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images_120/"
RESULTS_DIR = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/results/"
FILTER_CATALOG = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog2.fits"
# FILTER_CATALOG = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_le_120x120.fits" # Crop to [53,53]
# FILTER_CATALOG = "/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/split_images/catalog_gt_120x120.fits" # Crop to [121,121]

RUN_DIR = Path(BASE_DIR) / f"run{RUN_NUMBER}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_SPECS, EXPERIMENT_NUMBERS = build_model_experiment_plan_v2(
    model_name=MODEL_NAME,
    experiment_stage=EXPERIMENT_STAGE,
    baseline_experiment_number=BASELINE_EXPERIMENT_NUMBER,
    simclr_filters=SIMCLR_FILTERS,
)

print(f"Active experiment stage: {EXPERIMENT_STAGE}")
print_experiment_specs_v2({number: EXPERIMENT_SPECS[number] for number in EXPERIMENT_NUMBERS})


Active experiment stage: variants
_9: baseline _3 with batch_size=512 baseline=_3
_10: baseline _3 with batch_size=128 baseline=_3
_11: baseline _3 with final_layer=arcsinh and transform=arcsinh baseline=_3
_12: baseline _3 with filters=[g,r,i,z] baseline=_3
_13: baseline _3 with latent_dim=128 baseline=_3
_14: baseline _3 with latent_dim=256 baseline=_3
_15: baseline _3 with latent_dim=1024 baseline=_3
_16: baseline _3 with base_channel_size=16 baseline=_3
_17: baseline _3 with base_channel_size=64 baseline=_3
_18: baseline _3 with crop_to=[100,100] baseline=_3


In [5]:
# Optional: extend the selected model's experiment specs by adding entries here.
# This example adds _19 as a longer variant of the selected model's _7 baseline.
EXTRA_EXPERIMENTS = {
    # 19: {
    #     "description": f"{MODEL_NAME} _7 with epochs=40",
    #     "baseline_experiment": 7,
    #     "overrides": {"epochs": 40},
    # },
}

if EXTRA_EXPERIMENTS:
    EXPERIMENT_SPECS.update(
        build_extra_experiment_specs_v2(
            EXTRA_EXPERIMENTS,
            EXPERIMENT_SPECS,
            default_baseline_experiment=BASELINE_EXPERIMENT_NUMBER,
        )
    )
    EXPERIMENT_NUMBERS = sorted(set(EXPERIMENT_NUMBERS).union(int(number) for number in EXTRA_EXPERIMENTS))
    print_experiment_specs_v2({number: EXPERIMENT_SPECS[number] for number in EXPERIMENT_NUMBERS})


In [6]:
create_training_files_v2(
    run_number=RUN_NUMBER,
    filter_catalog=FILTER_CATALOG,
    crop_to=[120, 120],
    output_dir=RUN_DIR,
    data_dir=DATA_DIR,
    results_dir=RESULTS_DIR,
    model_name=MODEL_NAME,
    batch_size=256,
    epochs=20,
    experiment_specs=EXPERIMENT_SPECS,
    experiment_numbers=EXPERIMENT_NUMBERS,
)


Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_9.toml
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_9.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_10.toml
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_10.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_11.toml
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_11.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_12.toml
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_12.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_13.toml
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_13.sh
Created /work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/train11_14.toml


In [7]:
submit_training_jobs_v2(RUN_NUMBER, EXPERIMENT_NUMBERS, base_directory=BASE_DIR)


Submitting: sbatch train11_9.sh
Submitted batch job 20354817
Submitting: sbatch train11_10.sh
Submitted batch job 20354818
Submitting: sbatch train11_11.sh
Submitted batch job 20354819
Submitting: sbatch train11_12.sh
Submitted batch job 20354820
Submitting: sbatch train11_13.sh
Submitted batch job 20354821
Submitting: sbatch train11_14.sh
Submitted batch job 20354822
Submitting: sbatch train11_15.sh
Submitted batch job 20354823
Submitting: sbatch train11_16.sh
Submitted batch job 20354824
Submitting: sbatch train11_17.sh
Submitted batch job 20354825
Submitting: sbatch train11_18.sh
Submitted batch job 20354826


In [9]:
create_infer_scripts_batch_v2(RUN_NUMBER, EXPERIMENT_NUMBERS, base_directory=BASE_DIR)


Created infer scripts for train11_1
Created infer scripts for train11_2
Created infer scripts for train11_3
Created infer scripts for train11_4
Created infer scripts for train11_5
Created infer scripts for train11_6
Created infer scripts for train11_7
Created infer scripts for train11_8


[(PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_1.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_1.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_2.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_2.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_3.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_3.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_4.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_4.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_5.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/infer11_5.sh')),
 (PosixPath('/w

In [10]:
submit_infer_jobs_v2(RUN_NUMBER, EXPERIMENT_NUMBERS, base_directory=BASE_DIR)


Submitting: sbatch infer11_1.sh
Submitted batch job 20334398
Submitting: sbatch infer11_2.sh
Submitted batch job 20334400
Submitting: sbatch infer11_3.sh
Submitted batch job 20334401
Submitting: sbatch infer11_4.sh
Submitted batch job 20334402
Submitting: sbatch infer11_5.sh
Submitted batch job 20334403
Submitting: sbatch infer11_6.sh
Submitted batch job 20334404
Submitting: sbatch infer11_7.sh
Submitted batch job 20334405
Submitting: sbatch infer11_8.sh
Submitted batch job 20334406


In [11]:
create_udb_scripts_batch_v2(RUN_NUMBER, EXPERIMENT_NUMBERS, base_directory=BASE_DIR)


Created UDB scripts for infer11_1
Created UDB scripts for infer11_2
Created UDB scripts for infer11_3
Created UDB scripts for infer11_4
Created UDB scripts for infer11_5
Created UDB scripts for infer11_6
Created UDB scripts for infer11_7
Created UDB scripts for infer11_8


[(PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_1.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_1.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_2.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_2.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_3.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_3.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_4.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_4.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_5.toml'),
  PosixPath('/work/hdd/bemi/dmiura/data_downloads/tng100_snap72/hyrax_runs/run11/udb11_5.sh')),
 (PosixPath('/work/hdd/bemi/dmiura/

In [12]:
submit_udb_jobs_v2(RUN_NUMBER, EXPERIMENT_NUMBERS, base_directory=BASE_DIR)


Submitting: sbatch udb11_1.sh
Submitted batch job 20335603
Submitting: sbatch udb11_2.sh
Submitted batch job 20335604
Submitting: sbatch udb11_3.sh
Submitted batch job 20335605
Submitting: sbatch udb11_4.sh
Submitted batch job 20335606
Submitting: sbatch udb11_5.sh
Submitted batch job 20335607
Submitting: sbatch udb11_6.sh
Submitted batch job 20335608
Submitting: sbatch udb11_7.sh
Submitted batch job 20335609
Submitting: sbatch udb11_8.sh
Submitted batch job 20335610
